## 1. Load the Dataset

The baseline model notebook starts from the original SMS Spam Collection dataset.

The raw dataset contains:
- `v1` → message label
- `v2` → SMS message
- extra unnamed columns → empty columns that will be removed

We will recreate the required preprocessing pipeline in this notebook so that the notebook can run independently.

In [13]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/spam.csv",
    encoding="latin-1"
)

df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


## 2. Clean the Raw Dataset

The original dataset contains several unnamed columns that do not contain useful information.

We will:
1. Keep only the label and message columns.
2. Rename them to `label` and `message`.
3. Convert the labels into numerical values:
   - `ham` → `0`
   - `spam` → `1`

In [14]:
df = df[["v1", "v2"]]

df = df.rename(columns={
    "v1": "label",
    "v2": "message"
})

df["label"] = df["label"].map({
    "ham": 0,
    "spam": 1
})

df.head()

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


## 3. Recreate the Text Preprocessing Pipeline

The model should not receive raw SMS messages directly.

We will apply the preprocessing steps learned previously:

1. Convert text to lowercase
2. Remove punctuation
3. Tokenize the text
4. Remove stopwords
5. Apply stemming
6. Join the processed tokens back into text

The resulting text will be stored in `transformed_text`.

In [15]:
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words("english"))
ps = PorterStemmer()


def transform_text(text):
    text = text.lower()

    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    words = text.split()

    words = [
        word
        for word in words
        if word not in stop_words
    ]

    words = [
        ps.stem(word)
        for word in words
    ]

    return " ".join(words)

In [16]:
df["transformed_text"] = df["message"].apply(transform_text)

df[["message", "transformed_text"]].head()

,message,transformed_text
0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entri wkli comp win fa cup final tkt st m...
3,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


## 4. Convert Text into TF-IDF Features

Machine learning models cannot directly understand text.

TF-IDF converts each processed SMS message into a numerical vector.

Each column represents a word from the vocabulary, and each value represents how important that word is to a particular message.

We will use the same TF-IDF approach explored in the previous notebook.

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X = tfidf.fit_transform(df["transformed_text"])

X.shape

(5572, 6221)

## 5. Create the Target Variable

`X` contains our numerical text features.

`y` contains the target labels:

- `0` → ham
- `1` → spam

In [18]:
y = df["label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nLabel distribution:")
print(y.value_counts())

X shape: (5572, 6221)
y shape: (5572,)

Label distribution:
label
0    4825
1     747
Name: count, dtype: int64


## 6. Create the Train/Test Split

We will divide the dataset into:

- 80% training data
- 20% testing data

`stratify=y` preserves the original ham/spam distribution in both sets.

`random_state=42` makes the split reproducible.

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [20]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (4457, 6221)
X_test : (1115, 6221)
y_train: (4457,)
y_test : (1115,)


In [21]:
print("Training labels:")
print(y_train.value_counts())

print("\nTesting labels:")
print(y_test.value_counts())

Training labels:
label
0    3859
1     598
Name: count, dtype: int64

Testing labels:
label
0    966
1    149
Name: count, dtype: int64
